In [ ]:
# from sklearn.linear_model import Lasso
# from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
# from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# from sklearn.pipeline import Pipeline
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
# from sklearn.feature_selection import SelectFromModel
# from sklearn.ensemble import RandomForestRegressor
# import numpy as np
# import pandas as pd
# import joblib

In [26]:
# load the dataset using the provided URL
url = "https://raw.githubusercontent.com/arunk13/MSDA-Assignments/master/IS607Fall2015/Assignment3/student-mat.csv"
data = pd.read_csv(url, sep=';')

# display the first few rows of the dataset
data.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [ ]:
# ================= IMPORTS =================
import pandas as pd
import numpy as np
import pickle  

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ================= FEATURE ENGINEERING =================
data['G_avg'] = (data['G1'] + data['G2']) / 2
data['G_ratio'] = data['G2'] / (data['G1'] + 1)

# ================= TARGET & FEATURES =================
y = data['G3']

features = [
    'school', 'sex', 'age', 'famsize',
    'studytime', 'failures',
    'schoolsup', 'famsup', 'paid', 'activities',
    'higher', 'internet',
    'absences',
    'G1', 'G2',
    'G_avg', 'G_ratio'
]

X = data[features]

# ================= SAVE FEATURES =================
with open("features.pkl", "wb") as f:
    pickle.dump(features, f)

# ================= TRAIN TEST SPLIT =================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# ================= COLUMN TYPES =================
numeric_cols = ['age', 'studytime', 'failures', 'absences', 'G1', 'G2', 'G_avg', 'G_ratio']

categorical_cols = [
    'school', 'sex', 'famsize',
    'schoolsup', 'famsup', 'paid', 'activities',
    'higher', 'internet'
]

# ================= PREPROCESSING =================
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

# ================= PIPELINE =================
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ))
])

# ================= HYPERPARAMETER TUNING =================
params = {
    'regressor__n_estimators': [200, 300],
    'regressor__max_depth': [None, 10, 20],
    'regressor__min_samples_split': [2, 5]
}

grid = GridSearchCV(pipeline, params, cv=5, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)

best_pipeline = grid.best_estimator_

print("Best Params:", grid.best_params_)

# ================= CROSS VALIDATION =================
cv_score = cross_val_score(best_pipeline, X, y, cv=5, scoring='r2').mean()
print("Cross Validation R2:", cv_score)

# ================= EVALUATION =================
y_pred = best_pipeline.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\nFinal Model Performance:")
print("MSE:", mse)
print("MAE:", mae)
print("R2:", r2)

# ================= SAVE MODEL =================
with open("model.pkl", "wb") as f:
    pickle.dump(best_pipeline, f)

print("Model saved successfully!")

# ================= FEATURE IMPORTANCE =================
model = best_pipeline.named_steps['regressor']
preprocessor = best_pipeline.named_steps['preprocessor']

num_features = numeric_cols
cat_features = preprocessor.named_transformers_['cat'] \
    .named_steps['encoder'] \
    .get_feature_names_out(categorical_cols)

all_features = list(num_features) + list(cat_features)

importances = model.feature_importances_

feature_importance_df = pd.DataFrame({
    'feature': all_features,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print("\nTop Feature Importance:\n")
print(feature_importance_df.head(10))


Best Params: {'regressor__max_depth': 10, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 200}
Cross Validation R2: 0.8435764196647113

Final Model Performance:
MSE: 2.9777952221477695
MAE: 1.0306412885513048
R2: 0.8547774514626307
Model saved successfully!

Top Feature Importance:

           feature  importance
5               G2    0.512188
6            G_avg    0.283347
3         absences    0.111075
7          G_ratio    0.027488
0              age    0.021004
20   activities_no    0.005175
21  activities_yes    0.004963
4               G1    0.004210
1        studytime    0.004146
2         failures    0.003649
